In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab11.ipynb")

# Lab 11: Principal Component Analysis and Clustering


In lecture, we discussed how Principal Component Analysis (PCA) and clustering can be used to better understand the structure of a dataset. Both these techniques are part of *unsupervised learning* when we want to learn underlying relationship given only the data without target labels. PCA helps us reduce the dimensionality of high-dimensional data, while clustering helps us identify groups of similar observations. Specifically, PCA allows us to:

1. Understand the rank of the data. If k principal components capture almost all of the variance, then the data is roughly rank k.
2. Create 2D scatterplots of the data. Such plots are a rank 2 representation of our data and can help us visually identify clusters of similar observations.

In this lab, we will walk through examples involving both PCA and clustering. First, we will use the Iris dataset to perform PCA using the `np.linalg` package and interpret the resulting low-dimensional representations. Then, we will study clustering on several toy datasets, developing intuition for how different clustering methods behave and how to evaluate the number of clusters.

To receive credit for a lab, answer all questions correctly and submit before the deadline.

You must submit this assignment to Pensive by the on-time deadline, **Monday, April 27, 11:59 PM PT**. Please read the syllabus for the Slip Day policy. As a reminder, slip days are **not** applicable on labs. **We strongly encourage you to plan to submit your work to Pensieve several hours before the stated deadline.** This way, you will have ample time to contact staff for submission support.  

## Lab Walk-Through
In addition to the lab notebook, we have also released a prerecorded walk-through video of the lab in Spring 2026. We encourage you to reference this video as you work through the lab. Run the cell below to display the video.

All parts of the lab can be found on this [Playlist](https://www.youtube.com/playlist?list=PLQCcNQgUcDfpquQHlagj8ftM3o86wLP_W).

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("Pc69gXBAdSE")

### Collaboration Policy
Data science is a collaborative activity. While you may talk with others about this assignment, we ask that you **write your solutions individually**. If you discuss the assignment with others, please **include their names** in the cell below.

**Collaborators:** *list names here*

### Debugging Guide
If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Jupyter notebooks / Datahub, Pensieve, common SQL errors, and more.

In [ ]:
# Run this cell to set up your notebook; no further action is needed
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn import cluster

plt.style.use('fivethirtyeight') # Use plt.style.available to see more styles
sns.set()
sns.set_context("talk")
%matplotlib inline

<br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part 1: Principal Component Analysis

To begin, run the following cell to load the dataset into this notebook. 
* `iris_features` will contain a `NumPy` array of 4 attributes for 150 different plants (shape $150 \times 4$). 
* `iris_target` will contain the class of each plant. There are three classes of plants in the dataset: Iris-Setosa, Iris-Versicolour, and Iris-Virginica. The class names will be stored in `iris_target_names`.
* `iris_feature_names` will be a list of 4 names, one for each attribute in `iris_features`. 

In [ ]:
# Run this cell to set up; no further action is needed.

from sklearn.datasets import load_iris
iris_data = load_iris() # Loading the dataset

# Unpacking the data into arrays.
iris_features = iris_data['data']
iris_target = iris_data['target']
iris_feature_names = iris_data['feature_names']
iris_target_names = iris_data['target_names']

# Convert iris_target to string labels instead of int labels currently (0, 1, 2) for the classes.
iris_target = iris_target_names[iris_target]

Let's explore the data by creating a scatter matrix of our iris features. To do this, we'll create 2D scatter plots for every possible pair of our four features. This should result in six total scatter plots in our scatter matrix with the classes labeled in distinct colors for each plot.

In [ ]:
# Run this cell to see the plot; no further action is needed.
fig = plt.figure(figsize=(14, 10))
plt.suptitle("Scatter Matrix of Iris Features", fontsize=20)
plt.subplots_adjust(wspace=0.3, hspace=0.3)
for i in range(1, 4):
    for j in range(i):
        plot_index = 3*j + i
        plt.subplot(3, 3, plot_index)
        sns.scatterplot(x=iris_features[:, i],
                        y=iris_features[:, j],
                        hue=iris_target,
                       legend=(plot_index == 1))
        plt.xlabel(iris_feature_names[i])
        plt.ylabel(iris_feature_names[j])
        if plot_index == 1:
            plt.legend().remove()

# same legend for all subplots.
fig.legend(loc='lower left') 
fig.tight_layout()
plt.show()

<br>

---

### Question 1a

To apply PCA, we will first need to center the data so that the mean of each feature is 0. We will also go further and create **standardized** features, where we scale features such that they are centered with a standard deviation of 1. (We'll explore simple centered data in Part 2.)

Compute the column-wise mean of `iris_features` in the cell below and store it in `iris_mean`. Then, compute the column-wise standard deviation of `iris_features` and store it in `iris_std`. Each should be a `NumPy` array of 4 means/standard deviations, 1 for each feature. 

Finally, subtract `iris_mean` from `iris_features`, divide by `iris_std`, and save the result in `iris_standardized`.

**Hints:** 
* You may find ([`np.mean`'s documentation](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)), ([`np.average`'s documentation](https://numpy.org/doc/stable/reference/generated/numpy.average.html)), and/or ([`np.std`'s documentation](https://numpy.org/doc/stable/reference/generated/numpy.std.html)) helpful. Pay attention to the `axis` argument. 
* If you are confused about how `NumPy` deals with arithmetic operations between arrays of different shapes, [see this note about broadcasting](https://docs.scipy.org/doc/numpy/user/basics.broadcasting.html) for explanations/examples.

In [ ]:
iris_mean = ...
iris_std = ...
iris_standardized = ...
iris_mean, iris_std

In [ ]:
grader.check("q1a")

<br/>

---

**PC dimensionality.**
Each principal component, as described in lecture, is $d \times 1$, where $d$: number of features in original $n \times d$ matrix $\mathbb{X}$.

<img src="images/PCA.png" style="display: block; margin: auto;" width="400" alt = "Diagram illustrating matrix dimensions in Principal Component Analysis (PCA). An input data matrix X of size n by d (n samples, d features) is multiplied by a matrix of the top k principal components V (d by k), resulting in a transformed matrix Z of size n by k. The figure emphasizes that each principal component is a vector of length d and that PCA projects the original data into a lower-dimensional space with k components." >

Originally, columns of $\mathbb{X}$ serve as a basis (axes) of a coordinate space for visualization of our data. PCs become a basis for a *new* coordinate system, and are taken along the direction of maximum variance across *all* features. Thus, the dimension of each PC is number of features.

<br/>

---

**(Bonus) PCA dimensionality intuition from SVD.** Remember in the lecture PCA involves in decomposing $\mathbb{X}$ such that

$$
\mathbb{X} = US V^T
$$


Let us first check the dimensions of each matrix. We should also pay special attention to the structure of $S$.



In [ ]:
u, s, vt = np.linalg.svd(iris_standardized, full_matrices=False) # SOLUTION
print(f"Dimensions of U: {u.shape}")
print(f"S:\n {s * np.identity(4)}")
print(f"Dimensions of V Transpose: {vt.shape}")

Notice that $S$ is a diagonal matrix whose diagonal entries are called singular values.

<br/>

---

### Question 1b

What can we learn from the singular values in $S$?

To answer this question, let's take a step back and recap what we know about variance. For a given feature, say sepal length, the **variance** associated with it can be calculated as follows:

$$ \text{Var} (\mathbb{X}_\text{sepal length}) = \frac{1}{n} \sum_{i=1}^n (x_{i, \text{sepal length}} - \bar{x}_\text{sepal length})^2 $$

where $x_{i, \text{sepal length}}$ refers to the sepal length of the $i$-th iris, and $\bar{x}_\text{sepal length}$ is the average sepal length across all $n$ irises.

As discussed in lecture, we define the **total variance** of a dataset $\mathbb{X}$ to be the sum of variances of each of the features.

In this case, we have:

$$ \text{Var} (\mathbb{X}) = \text{Var} (\mathbb{X}_\text{sepal length}) + \text{Var} (\mathbb{X}_\text{sepal width})+ \text{Var} (\mathbb{X}_\text{petal length}) + \text{Var} (\mathbb{X}_\text{petal width})$$

The singular values in `s` provide us with a convenient way by which we can understand the total variance in $\mathbb{X}$. Specifically, the total variance in the data is also equal to the *sum of the squares of the singular values divided by the number of data points*, that is:

$$\text{Var}(\mathbb{X}) = \frac{\sum_{i=1}^p{\sigma_i^2}}{n} = \sum_{i=1}^p \frac{\sigma_i^2}{n} = \sum_{i=1}^p \text{Variance captured by } i\text{-th PC}$$

where for data $\mathbb{X}$ with $n$ datapoints and $p$ features, $\sigma_i$ is the singular value corresponding to the $i$-th principal component, and $\text{Var}(\mathbb{X})$ is the total variance of the data. 

Compute the total variance of our data below by summing the square of each singular value in `s` and dividing the result by the total number of data points. Store the result in the variable `iris_total_variance`.

In [ ]:
iris_total_variance = ...

print("iris_total_variance: {:.3f} should approximately equal the sum of the feature variances: {:.3f}"
      .format(iris_total_variance, np.sum(np.var(iris_standardized, axis=0))))

In [ ]:
grader.check("q1b")

As you can see, `iris_total_variance` equals the sum of the standardized feature variances.

<br/>

---

### Question 2a

Let's now use only the first two principal components to see what a 2D version of our iris data looks like.

First, construct the 2D version of the iris data by multiplying our `iris_standardized` array with the first two right singular vectors in $V$. Because the first two right singular vectors are directions for the first two principal components, this will project the iris data down from a 4D subspace to a 2D subspace.

**Hints:**
* To matrix-multiply two `NumPy` arrays, use `@` or `np.dot`. In case you're interested, [the matmul documentation](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) contrasts the two methods.
* Note that in Question 1b, you computed `vt` ($\mathbb{X} = US V^T$). The first two right singular vectors in $V$ will be the two rows of `vt`, [transposed (documentation can be found here)](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.T.html) to be column vectors instead of row vectors.  
* Since we want to obtain a 2D version of our iris dataset, the shape of `iris_2d` should be $150 \times 2$.

In [ ]:
iris_2d = ...
iris_2d.shape

In [ ]:
grader.check("q2a")

<br/>

Now, run the cell below to create the scatter plot of our 2D version of the iris data, `iris_2d`.

In [ ]:
# Run this cell to see the plot; no further action is needed.
plt.figure(figsize = (5, 5))
plt.title("PC2 vs. PC1 for Iris Data")
plt.xlabel("Iris PC1")
plt.ylabel("Iris PC2")
sns.scatterplot(x = iris_2d[:, 0], y = iris_2d[:, 1], hue = iris_target);

<!-- BEGIN QUESTION -->

<br/>

---

### Question 2b

What do you observe about the plot above? If you were given a point in the subspace defined by PC1 and PC2, how well would you be able to classify the point as one of the three Iris types?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/>

---
### Question 2c

What proportion of the total variance is accounted for when we project the iris data into two dimensions? Compute this quantity in the cell below by dividing the variance captured by the first two singular values in `s` by the `iris_total_variance` you calculated previously. Store the result in `iris_2d_variance`.

In [ ]:
iris_2d_variance = ...
iris_2d_variance

In [ ]:
grader.check("q2c")

Most of the variance in the data is explained by the two-dimensional projection!

<br/>

---

### Question 3

Next, we will create a ([**scree plot**](https://en.wikipedia.org/wiki/Scree_plot)) to visualize the weight of each principal component. In the cell below, create a scree plot by making a line plot of the variance captured by each principal component vs. the principal component number (1st, 2nd, 3rd, or 4th). Your graph should look similar to the image below:

**Hint:** Be sure to label your axes appropriately! You may find [`plt.xticks()`'s documentation](https://matplotlib.org/3.5.0/api/_as_gen/matplotlib.pyplot.xticks.html) helpful for formatting.

<img src="images/scree.png" width="400px" alt = "Scree plot of Iris principal components showing a sharp drop in explained variance from PC1 to PC2 and very little variance in PC3 and PC4."/>

In [ ]:
...

<br/>

---

## [Tutorial] Biplots

Finally, we will analyze the ([**biplot**](https://ds100.org/course-notes/pca/#biplots)) to understand how each feature contributes to the first two principal components. We do this by plotting the **directions**, or rows of $V^T$, which indicate how a feature “correlates” with each respective principal component. 

Note: The direction of each arrow is not unique up to sign, so an arrow and its opposite direction may both be valid.


Run the below cell to generate the biplot for the Iris dataset. Based on the directions plotted in the above biplot, what can you say about how each feature contributes to PC1 and PC2?

In [ ]:
# Run this cell to plot a biplot; no further action is needed.
import random
random.seed(42)
cp = sns.color_palette()[1:] # Skip blue

plt.figure(figsize = (7, 7))

# First plot each datapoint in terms of the first two principal components.
sns.scatterplot(x = iris_2d[:, 0], y = iris_2d[:, 1]);

# Next, plot the loadings for PC1 and PC2.
dir1, dir2 = vt[0,:], vt[1,:]
# Just plotting the 2 arrows corresponding to 'sepal_width' and 'petal_length' 
for i, feature in enumerate(['', 'sepal width', 'petal length', '']):
    plt.arrow(0, 0,
              dir1[i], dir2[i],
              head_width=0.2, head_length=0.2, color=cp[i])
    plt.text(dir1[i] * 1+0.1*random.random(), # jitter
             dir2[i] * 1+0.1*random.random(), 
             feature, fontsize=18, color=cp[i],
             backgroundcolor=(1,1,1,0.6))

plt.title("Iris Biplot")
plt.xlabel("Iris PC1")
plt.ylabel("Iris PC2")
plt.show()

<br>

---

## Question 4

Based on the principal components plotted in the above biplot, fill in the blanks for each of the following statements:

Q4a. `sepal width` looks to be ___________ with PC1.<br/>
Q4b. `sepal width` looks to be ___________ with PC2.<br/>
Q4c. `petal length` looks to be ___________ with PC1.<br/>
Q4d. `petal length` looks to be ___________ with PC2.

Note that we have displayed the arrows for all the features in the dataset, though you only need to look at the labeled arrows to answer the question (green - sepal width, red - petal length).

You should assign each variable (e.g., `q4a`) to`'A'`, `'B'`, `'C'` corresponding to the below:

A. positively correlated<br/>
B. negatively correlated <br/>
C. uncorrelated

In some cases, it may be difficult to draw the line between positively/negatively correlated and uncorrelated. The autograder tests will accept all reasonable answers if it is ambiguous.

**Note:** This question is not included in the lab walkthrough

In [ ]:
q4a = ...
q4b = ...
q4c = ...
q4d = ...

In [ ]:
grader.check("q4")

## Part 2 Clustering
In the second part of this lab, we will see how different clustering algorithms work on a few toy datasets.

<br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Toy Data 1: Balanced Clusters

Let's begin with a toy dataset with three groups that are completely separated in a 2D space. All groups/clusters have the same number of points, and the variance in all groups is the same.

In [ ]:
# Run this cell to plot the datapoints.
np.random.seed(1337)

c1 = np.random.normal(size = (25, 2))
c2 = np.array([2, 8]) + np.random.normal(size = (25, 2))
c3 = np.array([8, 4]) + np.random.normal(size = (25, 2))

toy_datapoints_1 = np.vstack((c1, c2, c3))

sns.scatterplot(x = toy_datapoints_1[:, 0], y = toy_datapoints_1[:, 1]);

Below, we create a `cluster.KMeans` object ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)) which implements the K-Means algorithm.

In [ ]:
# Plotting K-Means output.

def plot_k_means(data, classifier):
    palette = ['orange', 'brown', 'dodgerblue', 'red']
    sns.scatterplot(x = data[:, 0], y = data[:, 1], 
                hue = classifier.labels_, 
                palette = palette[:classifier.n_clusters])
    sns.scatterplot(x = classifier.cluster_centers_[:, 0], 
                    y = classifier.cluster_centers_[:, 1],
                    color = 'green', marker = 'x', s = 150, linewidth = 3);

In [ ]:
# Run this cell to plot the K-Means cluster.

kmeans = cluster.KMeans(n_clusters = 3, random_state = 42).fit(toy_datapoints_1)
plot_k_means(toy_datapoints_1, kmeans)

We observe that K-Means is able to accurately pick out the three initial clusters. 

<br/>

---

## Question 5: Initial Centers

In the previous example, the K-Means algorithm was able to accurately find the three initial clusters. However, changing the starting centers for K-Means can change the final clusters that K-Means gives us. For this question, we'll initialize the algorithms with `[0, 1]`, `[1, 1]`, and `[2, 2]`. A visualization is shown below: 

In [ ]:
sns.scatterplot(x = [0, 1, 2], 
                y = [1, 1, 2],
                color = 'green', marker = 'x', s = 150, linewidth = 3);

Change the initial centers to the points `[0, 1]`, `[1, 1]`, and `[2, 2]`, and fit a `cluster.KMeans` object ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)) called `kmeans_q1` on the toy dataset from the previous example. Keep the `random_state` parameter as 42, `n_clusters` parameter as 3, and `n_init` as 1.

**Hint:** You will need to change the `init` parameter in `cluster.KMeans`. Read the ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)) for more info!

In [ ]:
kmeans_q1 = ...

In [ ]:
grader.check("q5")

Running the K-Means algorithm with these centers gives us a different result from before, and this particular run of K-Means was unable to accurately find the three initial clusters.

In [ ]:
plot_k_means(toy_datapoints_1, kmeans_q1)

<br>

**Note**: Instead of using predetermined centers, one option is to use `init="random"`, and `sklearn` will randomly select initial centers. As shown above, the clustering result varies significantly based on your initial centers. Therefore, `init="random"` is often combined with `n_init` of greater than 1. `sklearn` will then run `n_init` number of times with different randomly-generated initial centers and return the best result. In our case, we are interested in specific initial centers, so we set `n_init=1`.

<br/>

<hr style="border: 1px solid #fdb515;" />

## Toy Data 2: Clusters of Different Sizes

Sometimes, K-Means will have a difficult time finding the "correct" clusters even with ideal starting centers. For example, consider the data below.

In [ ]:
# Run this cell to plot the datapoints.
np.random.seed(1337)

c1 = 0.5 * np.random.normal(size = (25, 2)) 
c2 = np.array([10, 10]) + 3 * np.random.normal(size = (475, 2))
toy_datapoints_2 = np.vstack((c1, c2))
sns.scatterplot(x = toy_datapoints_2[:, 0], y = toy_datapoints_2[:, 1]);

There are two groups of data with different **variability** (i.e., spread) and **number of data points**. The smaller group has both smaller variability and fewer data points, and the larger of the two groups is more diffuse and populated.

<br/>

---

## Question 6a: K-Means

Fit a `cluster.KMeans` object ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)) called `kmeans_q2a` on the dataset above with two clusters and a `random_state` parameter of 42. 

In [ ]:
kmeans_q2a = ...

In [ ]:
grader.check("q6a")

<br/>

As seen below, K-Means is unable to find the two initial clusters on the bottom left and top right. Cluster 1 (in red) is not centered at (0, 0), and it contains points that should belong in the other cluster. Recall that K-Means attempts to minimize inertia, so it makes sense that points near the bottom left of cluster 0 would prefer to be in cluster 1. If these points were in cluster 0 instead, then the resulting cluster assignments would have a larger distortion.

In [ ]:
# Run this cell to plot the kmeans cluster.

plot_k_means(toy_datapoints_2, kmeans_q2a)

<br/>

---

## Hierarchical Agglomerative Clustering: The Linkage Criterion

As shown in lecture 26, hierarchical agglomerative clustering works better for this task as long as we choose the right definition of distance between two clusters. Recall that hierarchical clustering starts with every data point in its own cluster and iteratively joins the two closest clusters until there are $k$ clusters remaining. However, the "distance" between the two clusters is ambiguous. 

In the lecture, we used the maximum distance between a point in the first cluster and a point in the second as this notion of distance (Complete Linkage hierarchical clustering), but there are other ways to define the distance between two clusters. 

Our choice of definition for the distance is sometimes called the "linkage criterion." We will discuss three linkage criteria, each of which is a different definition of "distance" between two clusters:

- **Complete linkage** defines the distance between two clusters as the **maximum** distance across all possible pairs consisting of a point from the first cluster and a point from the second cluster.
- **Single linkage** defines the distance between two clusters as the **minimum** distance across all possible pairs consisting of a point from the first cluster and a point from the second cluster.
- **Average linkage** defines the distance between two clusters as the **average** distance across all possible pairs consisting of a point from the first cluster and a point from the second cluster.

Below, we fit a `cluster.AgglomerativeClustering` object ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)) called `agg_complete` on the dataset above with two clusters, using the **complete linkage criterion**.

In [ ]:
# Run this cell to perform agglomerative clustering on the data.
agg_complete = cluster.AgglomerativeClustering(n_clusters = 2, linkage = 'complete').fit(toy_datapoints_2)

Below, we visualize the results:

In [ ]:
# Run this cell to plot the agglomerative clusters.
sns.scatterplot(x = toy_datapoints_2[:, 0], y = toy_datapoints_2[:, 1], hue = agg_complete.labels_);

It looks like complete linkage agglomerative clustering has the same issue as K-Means! The bottom left cluster found by complete linkage agglomerative clustering includes points from the top right cluster. However, we can remedy this by picking a different linkage criterion.

<br/>

---

## Question 6b: Agglomerative Clustering

Now, use the **single linkage criterion** to fit a `cluster.AgglomerativeClustering` object ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)) called `agg_single` on the dataset above with two clusters.


In [ ]:
agg_single = ...

In [ ]:
grader.check("q6b")

Finally, we see from the scatterplot on the left that single linkage agglomerative clustering is able to find the two initial clusters. We also plot a dendrogram to visualize the merging hierarchy.

In [ ]:
from scipy.cluster.hierarchy import dendrogram
def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram
    # Create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    distance = np.arange(model.children_.shape[0])

    linkage_matrix = np.column_stack(
        [model.children_, distance, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.title("Agglomerative Cluster")
sns.scatterplot(x = toy_datapoints_2[:, 0], y = toy_datapoints_2[:, 1], hue = agg_single.labels_);

plt.subplot(122)
plt.title("Dendrogram")
plot_dendrogram(agg_single, truncate_mode="level", p=3)
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()

You might be curious why single linkage "works" while complete linkage does not in this scenario; we will leave this as an exercise for students who are interested.

---
Let’s return to toy_datapoints_1 for now. It turns out that there are several other ways to decide how many clusters to use. Below, we introduce two of them: Silhouette Score and Elbow Plot.

## Silhouette Score

How many clusters $k$ should we use? We can use silhouette score and silhouette plot to determine this. [The following code is taken from the documentation here](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html#). Note that we are using an older version of `sklearn` (1.2). The most recent version, `sklearn` 1.4, requires a slightly different code.

In [ ]:
# Run this cell to generate the silhouette plots; no further action required.

from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.cm as cm

X = toy_datapoints_1
range_n_clusters = [2, 3, 4, 5, 6, 7, 8, 9]
inertia = []

plt.figure(figsize=(15, 8))
for n_clusters in range_n_clusters:
    atoy_datapoints_1 = plt.subplot(2, 4, range_n_clusters.index(n_clusters)+1)
    # The (n_clusters+1)*10 is for inserting blank space between silhouette
    # plots of individual clusters, to demarcate them clearly.
    atoy_datapoints_1.set_xlim([-0.1, 0.8])
    atoy_datapoints_1.set_ylim([0, len(X) + (n_clusters + 1) * 10])

    # Initialize the clusterer with n_clusters value and a random generator
    # seed of 42 for reproducibility.
    clusterer = cluster.KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = clusterer.fit_predict(X)
    inertia += [clusterer.inertia_]

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed clusters
    silhouette_avg = silhouette_score(X, cluster_labels)

    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        # Aggregate the silhouette scores for samples belonging to
        # cluster i, and sort them
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        color = cm.nipy_spectral(float(i) / n_clusters)
        atoy_datapoints_1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the silhouette plots with their cluster numbers at the middle
        atoy_datapoints_1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    atoy_datapoints_1.set_title("n_clusters = %d"% n_clusters, fontsize=10)
    atoy_datapoints_1.set_xlabel("silhouette coefficient values", fontsize=10)
    atoy_datapoints_1.set_ylabel("cluster label", fontsize=3)

    # The vertical line for average silhouette score of all the values
    atoy_datapoints_1.axvline(x=silhouette_avg, color="red", linestyle="--")

    atoy_datapoints_1.set_yticks([])  # Clear the yaxis labels / ticks
    atoy_datapoints_1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8])


plt.tight_layout()
plt.show()

<!-- BEGIN QUESTION -->

<br>

---

## Question 7

Based on the plots above, how many clusters $k$ would you use and why? There is no single correct answer.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br>

---

## Question 8

We have saved the inertia generated using different numbers of clusters above in the array `inertia` and used it to construct an elbow plot. Based on the plot, what number of clusters you would choose and why? There is no single correct answer.

In [ ]:
plt.plot(range_n_clusters, inertia)
plt.scatter(range_n_clusters, inertia)
plt.title("Elbow plot")
plt.xlabel("number of cluster")
plt.ylabel("inertia")
plt.xticks(range_n_clusters);

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations for finishing Lab 11, the last Lab for Data 100!

Best of luck on the final and your future data science endeavors!

### Course Content Feedback

If you have any feedback about this assignment or about any of our other weekly assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://docs.google.com/forms/d/e/1FAIpQLSelI7jXeKh6eYqWnpEsJVPSuRL0JWZ2yjLNbZJMxekUquDetA/viewform?usp=dialog). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

### Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Submit this file to the Lab 11 assignment on Pensieve. If you run into any issues when running this cell, feel free to check this [section](https://ds100.org/debugging-guide/autograder-gradescope/#why-does-the-last-grader-export-cell-fail-if-all-previous-tests-passed) in the Data 100 Debugging Guide.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)